In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import sys, os
sys.path.append(os.path.dirname(os.path.abspath('.')))

from sqlalchemy import text
from db.database import engine
from etl.constants import EUROPEAN_COUNTRIES, DECOUPLING_END_YEAR

# ================= CONFIG: the knobs =================
BASE_YEARS  = [1990, 2000]
LATEST_YEAR = DECOUPLING_END_YEAR  # 2023

# Normative weights (must sum to 1.0). Honesty raised to 0.40 - the thesis (real vs fake)
# carries the most weight; the rest share equally.
WEIGHTS = {
    'honesty':         0.40,   # consumption CO2 reduction MINUS fake-decoupling penalty (can be negative)
    'energy_clean':    0.15,   # absolute current grid cleanliness (rewards already-clean)
    'abs_consumption': 0.15,   # absolute consumption CO2/capita vs 2t target
    'prosperity':      0.15,   # absolute GDP/capita + positive-growth guard
    'co2_reduction':   0.15,   # territorial CO2/capita reduction (supporting)
}
assert abs(sum(WEIGHTS.values()) - 1.0) < 1e-9

# Sub-weights inside the prosperity component
PROSPERITY_SUB = {'abs_gdp': 0.6, 'growth_guard': 0.4}

# Honesty = blend of long-run and recent consumption reduction, minus fake penalty
HONESTY_BLEND = {'long': 0.6, 'recent': 0.4}
RECENCY_START = 2010

# Criterion-referenced scoring thresholds (fixed, cohort-independent)
# score_linear(value, zero, full): zero -> 0 pts, full -> 100 pts, clipped
SCORING = {
    'co2_reduction':         {'zero': 0.0,    'full': 60.0},   # territorial % reduction
    'cons_reduction':        {'zero': 0.0,    'full': 45.0},   # consumption % reduction (long-run, honesty)
    'cons_reduction_recent': {'zero': 0.0,    'full': 30.0},   # consumption % reduction since RECENCY_START
    'fake_penalty':          {'zero': 0.0,    'full': 60.0},   # divergence% (terr fell more than cons) -> penalty
    'abs_consumption':       {'zero': 16.0,   'full': 2.0},    # t/capita: 16t->0, 2t->100
    'energy_clean':          {'zero': 0.30,   'full': 0.03},   # kgCO2/kWh: dirty 0.30->0, clean 0.05->100
    'abs_gdp':               {'zero': 15000.0,'full': 60000.0},# $/capita: 15k->0, 60k->100
    'growth_guard':          {'zero': -30.0,  'full': 0.0},    # % growth: -30%->0, >=0 ->100 (guard)
}

# Momentum modifier (small, applied on top). full_swing_slope raised to 0.30 so it
# DISCRIMINATES (at 0.10 almost everyone hit the +3 cap and it added nothing).
MOMENTUM = {'window_start': 2010, 'full_swing_slope': 0.30, 'max_bonus': 3.0}

# Known data-quality distortions: flagged, and shown in a separate 'clean' ranking without them
DATA_CAVEAT = {
    'IRL': 'GDP inflated by multinational profit-shifting (real income ~ GNI*)',
    'LUX': 'cross-border workers inflate per-capita GDP & emissions; fuel-tourism legacy',
    'MLT': 'tiny island economy, sparse/volatile data',
}

# Decoupling verdicts from NB04 (coloring / face-validity)
GENUINE = ['DEU', 'SWE', 'ROU', 'EST', 'LTU', 'SVK', 'BGR', 'FIN', 'NOR']
FAKE    = ['CHE', 'BEL', 'DNK', 'FRA', 'GBR', 'BLR']
SPECIAL = ['POL', 'IRL', 'CYP', 'GEO']

def verdict_of(iso):
    if iso in GENUINE: return 'Genuine'
    if iso in FAKE:    return 'Fake'
    if iso in SPECIAL: return 'Special'
    return 'Other'

VERDICT_COLORS = {'Genuine': '#2ecc71', 'Fake': '#e74c3c',
                  'Special': '#f39c12', 'Other': '#95a5a6'}

print('Config loaded. Weights sum =', round(sum(WEIGHTS.values()), 4))

Config loaded. Weights sum = 1.0


In [2]:
q = """
    SELECT
        c.name AS country, c.iso_code, e.year,
        e.co2_per_capita,
        e.consumption_co2_per_capita,
        e.co2_per_unit_energy,
        e.gdp, e.population
    FROM emissions e
    JOIN countries c ON c.id = e.country_id
    WHERE c.iso_code = ANY(:countries)
    ORDER BY c.iso_code, e.year
"""
with engine.connect() as conn:
    df = pd.read_sql(text(q), conn, params={'countries': EUROPEAN_COUNTRIES})

df['gdp_per_capita'] = df['gdp'] / df['population']
print(f"Loaded {len(df)} rows, {df['iso_code'].nunique()} countries")

# Calibration: LAST NON-NULL value per country (GDP series ends ~2019, so a fixed
# recent window left it blank - groupby.last() skips NaN and always finds the real value).
g_last = df.sort_values('year').groupby('iso_code')

print("\nLatest co2_per_unit_energy (kgCO2/kWh) - calibrate 'energy_clean':")
print(g_last['co2_per_unit_energy'].last().describe().round(3).to_string())
print("\nLatest gdp_per_capita - calibrate 'abs_gdp':")
print(g_last['gdp_per_capita'].last().describe().round(0).to_string())
print("\nLatest consumption_co2_per_capita - calibrate 'abs_consumption':")
print(g_last['consumption_co2_per_capita'].last().describe().round(2).to_string())

2026-06-09 23:11:22,435 INFO sqlalchemy.engine.Engine select pg_catalog.version()
2026-06-09 23:11:22,435 INFO sqlalchemy.engine.Engine [raw sql] {}
2026-06-09 23:11:22,439 INFO sqlalchemy.engine.Engine select current_schema()
2026-06-09 23:11:22,440 INFO sqlalchemy.engine.Engine [raw sql] {}
2026-06-09 23:11:22,444 INFO sqlalchemy.engine.Engine show standard_conforming_strings
2026-06-09 23:11:22,444 INFO sqlalchemy.engine.Engine [raw sql] {}
2026-06-09 23:11:22,452 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-06-09 23:11:22,457 INFO sqlalchemy.engine.Engine SELECT pg_catalog.pg_class.relname 
FROM pg_catalog.pg_class JOIN pg_catalog.pg_namespace ON pg_catalog.pg_namespace.oid = pg_catalog.pg_class.relnamespace 
WHERE pg_catalog.pg_class.relname = %(table_name)s AND pg_catalog.pg_class.relkind = ANY (ARRAY[%(param_1)s, %(param_2)s, %(param_3)s, %(param_4)s, %(param_5)s]) AND pg_catalog.pg_table_is_visible(pg_catalog.pg_class.oid) AND pg_catalog.pg_namespace.nspname != %(nspname

In [3]:
def val_at(group, year, col):
    row = group[group['year'] == year]
    return np.nan if row.empty else row[col].values[0]

def val_latest(group, col, not_after=LATEST_YEAR):
    sub = group[(group['year'] <= not_after) & group[col].notna()]
    return np.nan if sub.empty else sub.sort_values('year')[col].values[-1]

def pct_reduction(base, latest):
    if pd.isna(base) or pd.isna(latest) or base == 0: return np.nan
    return (base - latest) / base * 100

def pct_growth(base, latest):
    if pd.isna(base) or pd.isna(latest) or base == 0: return np.nan
    return (latest - base) / base * 100

def recent_slope(group, col, start_year):
    sub = group[(group['year'] >= start_year) & group[col].notna()]
    if len(sub) < 4: return np.nan
    return np.polyfit(sub['year'].astype(float), sub[col].astype(float), 1)[0]

In [4]:
def score_linear(value, zero, full):
    # zero -> 0 pts, full -> 100 pts, clipped to [0, 100]
    if pd.isna(value): return np.nan
    return float(np.clip((value - zero) / (full - zero) * 100, 0, 100))

def momentum_bonus(slope):
    if pd.isna(slope): return 0.0
    raw = -slope / MOMENTUM['full_swing_slope'] * MOMENTUM['max_bonus']
    return float(np.clip(raw, -MOMENTUM['max_bonus'], MOMENTUM['max_bonus']))

def wavg_available(scores, weights):
    # weighted average over non-nan components, weights renormalized
    avail = {k: v for k, v in scores.items() if pd.notna(v)}
    if not avail: return np.nan
    wsum = sum(weights[k] for k in avail)
    return sum(scores[k] * weights[k] for k in avail) / wsum

In [5]:
SC = SCORING

def honesty_base_year(g, base_year):
    # earliest year >= base_year where BOTH consumption and territorial exist
    sub = g[(g['year'] >= base_year) &
            g['consumption_co2_per_capita'].notna() &
            g['co2_per_capita'].notna()]
    return int(sub['year'].min()) if not sub.empty else None

def first_from(g, col, start_year):
    sub = g[(g['year'] >= start_year) & g[col].notna()].sort_values('year')
    return sub[col].values[0] if not sub.empty else np.nan

def build_scores(df, base_year):
    rows = []
    for iso, g in df.groupby('iso_code'):
        g = g.sort_values('year')
        country = g['country'].iloc[0]

        # ---- raw metrics ----
        co2_base    = val_at(g, base_year, 'co2_per_capita')
        co2_latest  = val_latest(g, 'co2_per_capita')
        cons_latest = val_latest(g, 'consumption_co2_per_capita')
        gdp_base    = val_at(g, base_year, 'gdp_per_capita')
        gdp_latest  = val_latest(g, 'gdp_per_capita')
        en_latest   = val_latest(g, 'co2_per_unit_energy')

        co2_red  = pct_reduction(co2_base, co2_latest)       # territorial fell % (GLOBAL base)
        gdp_grw  = pct_growth(gdp_base, gdp_latest)
        if pd.notna(cons_latest) and pd.notna(co2_latest) and co2_latest != 0:
            gap_pct = (cons_latest - co2_latest) / co2_latest * 100
        else:
            gap_pct = np.nan

        # ---- HONESTY base year: fall back to earliest consumption year (e.g. Norway -> 2003) ----
        # measure BOTH consumption and territorial reduction from the SAME year so divergence is fair
        hby = honesty_base_year(g, base_year)
        if hby is not None:
            cons_base_h = val_at(g, hby, 'consumption_co2_per_capita')
            co2_base_h  = val_at(g, hby, 'co2_per_capita')
            cons_red    = pct_reduction(cons_base_h, cons_latest)        # long-run reduction
            co2_red_h   = pct_reduction(co2_base_h, co2_latest)
            divergence  = (co2_red_h - cons_red) if (pd.notna(co2_red_h) and pd.notna(cons_red)) else np.nan
        else:
            cons_red = divergence = np.nan

        # recent consumption reduction (since RECENCY_START) - tempers bubble-era baselines (e.g. Ireland)
        cons_recent_base = first_from(g, 'consumption_co2_per_capita', RECENCY_START)
        cons_red_recent  = pct_reduction(cons_recent_base, cons_latest)

        # ---- HONESTY: blend(long, recent) consumption reduction minus fake penalty (can go negative) ----
        s_long   = score_linear(cons_red,        SC['cons_reduction']['zero'],        SC['cons_reduction']['full'])
        s_recent = score_linear(cons_red_recent, SC['cons_reduction_recent']['zero'], SC['cons_reduction_recent']['full'])
        honesty_base = wavg_available({'long': s_long, 'recent': s_recent}, HONESTY_BLEND)
        penalty = score_linear(divergence, SC['fake_penalty']['zero'], SC['fake_penalty']['full'])
        penalty = 0.0 if pd.isna(penalty) else penalty
        s_honesty = (honesty_base - penalty) if pd.notna(honesty_base) else np.nan

        # ---- ENERGY: absolute current grid cleanliness ----
        s_energy = score_linear(en_latest, SC['energy_clean']['zero'], SC['energy_clean']['full'])

        # ---- ABSOLUTE consumption level ----
        s_abs = score_linear(cons_latest, SC['abs_consumption']['zero'], SC['abs_consumption']['full'])

        # ---- PROSPERITY: absolute GDP + positive-growth guard ----
        s_abs_gdp = score_linear(gdp_latest, SC['abs_gdp']['zero'], SC['abs_gdp']['full'])
        s_guard   = score_linear(gdp_grw,   SC['growth_guard']['zero'], SC['growth_guard']['full'])
        s_prosperity = wavg_available({'abs_gdp': s_abs_gdp, 'growth_guard': s_guard}, PROSPERITY_SUB)

        # ---- TERRITORIAL reduction (supporting, GLOBAL base) ----
        s_co2 = score_linear(co2_red, SC['co2_reduction']['zero'], SC['co2_reduction']['full'])

        comp = {'honesty': s_honesty, 'energy_clean': s_energy, 'abs_consumption': s_abs,
                'prosperity': s_prosperity, 'co2_reduction': s_co2}
        composite = wavg_available(comp, WEIGHTS)

        bonus = momentum_bonus(recent_slope(g, 'consumption_co2_per_capita', MOMENTUM['window_start']))
        final = composite + bonus if pd.notna(composite) else np.nan

        rows.append({
            'iso_code': iso, 'country': country, 'verdict': verdict_of(iso),
            'flag': '!' if iso in DATA_CAVEAT else '',
            'honesty_base_yr': hby,
            'co2_red_pct': co2_red, 'cons_red_pct': cons_red, 'cons_red_recent_pct': cons_red_recent,
            'gdp_grw_pct': gdp_grw, 'gap_pct': gap_pct, 'divergence': divergence,
            'cons_latest': cons_latest, 'gdp_latest': gdp_latest, 'en_latest': en_latest,
            's_honesty': s_honesty, 's_energy': s_energy, 's_abs': s_abs,
            's_prosperity': s_prosperity, 's_co2': s_co2,
            'composite': composite, 'momentum': bonus, 'final': final,
        })

    out = pd.DataFrame(rows)
    # gate: must have territorial reduction + absolute consumption
    out = out[out['final'].notna() & out['s_co2'].notna() & out['s_abs'].notna()].copy()
    return out.sort_values('final', ascending=False).reset_index(drop=True)

scores_2000 = build_scores(df, 2000)
scores_1990 = build_scores(df, 1990)

print(f"2000 base: {len(scores_2000)} countries scored")
print(f"1990 base: {len(scores_1990)} countries scored")

# Transparency: which countries used a fallback honesty base year?
for label, sdf in [(2000, scores_2000), (1990, scores_1990)]:
    diff = sdf[sdf['honesty_base_yr'].notna() & (sdf['honesty_base_yr'] != label)]
    if len(diff):
        print(f"\n[{label} base] honesty measured from a later year (consumption data starts late):")
        for _, r in diff.sort_values('honesty_base_yr').iterrows():
            print(f"   {r['iso_code']}: honesty base {int(r['honesty_base_yr'])}")

2000 base: 34 countries scored
1990 base: 34 countries scored

[2000 base] honesty measured from a later year (consumption data starts late):
   NOR: honesty base 2003

[1990 base] honesty measured from a later year (consumption data starts late):
   NOR: honesty base 2003


In [6]:
def show_ranking(scores, base_year, n=20, exclude_flagged=False):
    s = scores[scores['flag'] == ''] if exclude_flagged else scores
    cols = ['flag', 'iso_code', 'country', 'final', 'composite', 'momentum',
            's_honesty', 's_energy', 's_abs', 's_prosperity', 's_co2']
    disp = s.head(n)[cols].copy()
    disp.insert(0, 'rank', range(1, len(disp) + 1))
    num_cols = ['final', 'composite', 'momentum', 's_honesty', 's_energy', 's_abs', 's_prosperity', 's_co2']
    for c in num_cols:
        disp[c] = disp[c].round(1)
    tag = '  (excl. flagged)' if exclude_flagged else ''
    print(f"=== DECOUPLING INDEX (base {base_year}){tag} - Top {n} ===")
    print(disp.to_string(index=False))

# Clean ranking (no distorted economies) is the headline; full ranking shown after
show_ranking(scores_2000, 2000, exclude_flagged=True)
print()
show_ranking(scores_2000, 2000)
print()
show_ranking(scores_1990, 1990, exclude_flagged=True)

print("\nFlagged for data-quality caveats (excluded from the 'clean' ranking):")
for iso, why in DATA_CAVEAT.items():
    print(f"   {iso}: {why}")

=== DECOUPLING INDEX (base 2000)  (excl. flagged) - Top 20 ===
 rank flag iso_code        country  final  composite  momentum  s_honesty  s_energy  s_abs  s_prosperity  s_co2
    1           SWE         Sweden   82.6       80.5       2.1       82.4      88.5   72.5          82.8   72.9
    2           PRT       Portugal   82.2       80.6       1.6      100.0      58.9   81.8          57.8   72.3
    3           FIN        Finland   81.9       78.9       3.0       90.5      75.9   53.8          74.1   80.8
    4           NOR         Norway   80.5       77.5       3.0       84.7      85.2   63.8         100.0   41.5
    5           GBR United Kingdom   76.5       73.7       2.8       80.9      51.9   63.6          71.1   89.1
    6           EST        Estonia   72.6       69.6       3.0       87.0      53.7   48.5          59.5   70.4
    7           DNK        Denmark   68.1       65.8       2.3       57.3      56.3   54.9          87.6   87.2
    8           DEU        Germany   67.4

In [14]:
# Horizontal bar chart of the headline (clean) ranking - top 15
clean = scores_2000[scores_2000['flag'] == ''].head(15).copy()
clean = clean.sort_values('final')  # ascending -> highest bar on top

fig = px.bar(
    clean,
    x='final', y='country',
    orientation='h',
    color='final',
    color_continuous_scale='Tealgrn',
    text='final',
    title='Decoupling Index - Top 15 (2000 base, distorted economies excluded)',
    labels={'final': 'Decoupling score', 'country': ''},
    height=600,
)
fig.update_traces(texttemplate='%{text:.1f}', textposition='outside', cliponaxis=False)
fig.update_layout(coloraxis_showscale=False, xaxis_range=[0, 100],
                  margin=dict(l=120))
fig.show()




# Horizontal bar chart of the headline (clean) ranking - top 15
clean_2 = scores_1990[scores_1990['flag'] == ''].head(15).copy()
clean_2 = clean_2.sort_values('final')  # ascending -> highest bar on top

fig_2 = px.bar(
    clean_2,
    x='final', y='country',
    orientation='h',
    color='final',
    color_continuous_scale='Tealgrn',
    text='final',
    title='Decoupling Index - Top 15 (1990 base, distorted economies excluded)',
    labels={'final': 'Decoupling score', 'country': ''},
    height=600,
)
fig_2.update_traces(texttemplate='%{text:.1f}', textposition='outside', cliponaxis=False)
fig_2.update_layout(coloraxis_showscale=False, xaxis_range=[0, 100],
                  margin=dict(l=120))
fig_2.show()

In [8]:
r2000 = scores_2000.reset_index(drop=True).copy()
r1990 = scores_1990.reset_index(drop=True).copy()
r2000['rank_2000'] = range(1, len(r2000) + 1)
r1990['rank_1990'] = range(1, len(r1990) + 1)

merged = r2000[['iso_code', 'country', 'verdict', 'rank_2000']].merge(
    r1990[['iso_code', 'rank_1990']], on='iso_code', how='inner'
)
merged['shift'] = merged['rank_2000'] - merged['rank_1990']  # +ve = worse under 2000

fig = go.Figure()
for _, row in merged.iterrows():
    fig.add_trace(go.Scatter(
        x=['1990 base', '2000 base'],
        y=[row['rank_1990'], row['rank_2000']],
        mode='lines+markers+text',
        text=[row['iso_code'], row['iso_code']],
        textposition='middle right',
        line=dict(color=VERDICT_COLORS.get(row['verdict'], '#95a5a6')),
        showlegend=False
    ))
fig.update_layout(
    title='Ranking shift: 1990 vs 2000 baseline<br>'
          '<sup>Lines dropping left to right rode the Soviet-collapse windfall</sup>',
    yaxis=dict(title='Rank', autorange='reversed'),
    height=750
)
fig.show()

print("Most baseline-sensitive (rank worsens most under 2000 = inflated by 1990 collapse):")
print(merged.sort_values('shift', ascending=False)
      [['country', 'verdict', 'rank_1990', 'rank_2000', 'shift']].head(8).to_string(index=False))

Most baseline-sensitive (rank worsens most under 2000 = inflated by 1990 collapse):
    country verdict  rank_1990  rank_2000  shift
Netherlands   Other          9         22     13
   Slovakia Genuine          4         17     13
    Romania Genuine         17         28     11
    Ukraine   Other         21         31     10
    Belarus    Fake         30         34      4
    Germany Genuine          6         10      4
   Slovenia   Other         26         29      3
  Lithuania Genuine         24         27      3


In [9]:
top = scores_2000.head(15).copy()
contrib = pd.DataFrame({'iso_code': top['iso_code']})
pairs = [('honesty', 's_honesty'), ('co2_reduction', 's_co2'),
         ('abs_consumption', 's_abs'), ('energy_clean', 's_energy'), ('prosperity', 's_prosperity')]
for k, scol in pairs:
    contrib[k] = (top[scol].fillna(0) * WEIGHTS[k]).values

labels = {'honesty': 'Honesty (cons. reduction - fake penalty)', 'co2_reduction': 'Territorial CO2 reduction',
          'abs_consumption': 'Absolute consumption', 'energy_clean': 'Energy cleanliness (absolute)',
          'prosperity': 'Prosperity (GDP)'}
colors = {'honesty': '#1abc9c', 'co2_reduction': '#3498db', 'abs_consumption': '#9b59b6',
          'energy_clean': '#e67e22', 'prosperity': '#2c3e50'}

fig = go.Figure()
for k, _ in pairs:
    fig.add_trace(go.Bar(name=labels[k], x=contrib['iso_code'], y=contrib[k], marker_color=colors[k]))
fig.update_layout(
    barmode='relative', height=560,
    title='What drives each score? Weighted contributions (2000 base, Top 15)<br>'
          '<sup>Honesty can dip below zero for fake decouplers</sup>',
    yaxis_title='Points (weighted)', xaxis_title='Country'
)
fig.add_hline(y=0, line_color='black', line_width=1)
fig.show()

In [10]:
sc = scores_2000.copy()
fig = px.scatter(
    sc, 
    x='composite', 
    y='cons_latest',
    color='verdict', 
    text='iso_code',
    color_discrete_map=VERDICT_COLORS,
    title='Journey vs Destination (2000 base)<br>'
          '<sup>X = decoupling score | Y = consumption CO2/capita now. Top-right = clean AND high-scoring.</sup>',
    labels={'composite': 'Decoupling score', 'cons_latest': 'Consumption CO2 per capita (t, latest)'},
    height=680
)
fig.add_hline(y=2, line_dash='dash', line_color='green', annotation_text='2t target')
fig.update_traces(textposition='top center', marker=dict(size=11))
fig.update_yaxes(autorange='reversed')
fig.show()

In [11]:
# SENSITIVITY 1: Named weighting scenarios
comp_cols = {'honesty': 's_honesty', 'energy_clean': 's_energy', 'abs_consumption': 's_abs',
             'prosperity': 's_prosperity', 'co2_reduction': 's_co2'}

def rank_with_weights(scores_df, weights):
    def calc(row):
        return wavg_available({k: row[c] for k, c in comp_cols.items()}, weights)
    tmp = scores_df.copy()
    tmp['ctest'] = tmp.apply(calc, axis=1)
    tmp = tmp.sort_values('ctest', ascending=False).reset_index(drop=True)
    return {row['iso_code']: i + 1 for i, row in tmp.iterrows()}

# basis = CLEAN ranking (distorted economies excluded) - the headline leaderboard
base_clean = scores_2000[scores_2000['flag'] == ''].reset_index(drop=True)

scenarios = {
    'Normative':         WEIGHTS,
    'Equal':             {k: 0.20 for k in WEIGHTS},
    'Honesty-heavy':     {'honesty': 0.60, 'energy_clean': 0.10, 'abs_consumption': 0.10, 'prosperity': 0.10, 'co2_reduction': 0.10},
    'Destination-heavy': {'honesty': 0.20, 'energy_clean': 0.25, 'abs_consumption': 0.30, 'prosperity': 0.15, 'co2_reduction': 0.10},
    'Prosperity-heavy':  {'honesty': 0.20, 'energy_clean': 0.15, 'abs_consumption': 0.10, 'prosperity': 0.40, 'co2_reduction': 0.15},
    'Reduction-heavy':   {'honesty': 0.20, 'energy_clean': 0.15, 'abs_consumption': 0.10, 'prosperity': 0.15, 'co2_reduction': 0.40},
}

rank_table = {name: rank_with_weights(base_clean, w) for name, w in scenarios.items()}
sens = pd.DataFrame(rank_table)
scen_cols = list(scenarios.keys())
sens['best_rank']  = sens[scen_cols].min(axis=1)
sens['worst_rank'] = sens[scen_cols].max(axis=1)
sens['rank_range'] = sens['worst_rank'] - sens[scen_cols].min(axis=1)
sens = sens.sort_values('Normative')

print("Rank under each named weighting scheme (2000 base, clean):")
print(sens.head(12).to_string())

Rank under each named weighting scheme (2000 base, clean):
     Normative  Equal  Honesty-heavy  Destination-heavy  Prosperity-heavy  Reduction-heavy  best_rank  worst_rank  rank_range
PRT          1      4              1                  3                 6                4          1           6           5
SWE          2      1              3                  1                 2                1          1           3           2
FIN          3      3              2                  4                 3                2          2           4           2
NOR          4      2              4                  2                 1                6          1           6           5
GBR          5      5              5                  5                 5                3          3           5           2
EST          6      9              6                  9                11                7          6          11           5
DNK          7      6             11                  8    

In [12]:
# SENSITIVITY 2: Monte Carlo over the full weight space
# Draw N random weightings (uniform over the simplex) and watch the ranking.
# N=2000 is statistically plenty: a proportion from N draws has std err ~ sqrt(p(1-p)/N),
rng = np.random.default_rng(42)
N = 2000
comp_keys = list(WEIGHTS.keys())
iso_list  = base_clean['iso_code'].tolist()
rank_hist = {iso: [] for iso in iso_list}

for _ in range(N):
    w = dict(zip(comp_keys, rng.dirichlet(np.ones(len(comp_keys)))))
    for iso, r in rank_with_weights(base_clean, w).items():
        rank_hist[iso].append(r)

summary = []
for iso in iso_list:
    ranks = np.array(rank_hist[iso])
    summary.append({
        'iso_code': iso,
        'median_rank': int(np.median(ranks)),
        'best': int(ranks.min()), 'worst': int(ranks.max()),
        'pct_top5':  round((ranks <= 5).mean() * 100, 1),
        'pct_top10': round((ranks <= 10).mean() * 100, 1),
    })
mc = pd.DataFrame(summary).sort_values('median_rank').reset_index(drop=True)

print(f"Monte Carlo sensitivity - {N} random weightings (2000 base, clean):")
print(mc.head(12).to_string(index=False))

# Rank-distribution box plot for the contenders (narrow box = robust regardless of weights)
contenders = mc.head(12)['iso_code'].tolist()
dist = pd.DataFrame([{'iso_code': iso, 'rank': r}
                     for iso in contenders for r in rank_hist[iso]])
fig = px.box(
    dist, x='iso_code', y='rank',
    category_orders={'iso_code': contenders},
    title=f'Rank stability across {N} random weightings (2000 base, clean)<br>'
          '<sup>Narrow box = position holds no matter how you weight the components</sup>',
    labels={'rank': 'Rank', 'iso_code': ''}, height=520
)
fig.update_yaxes(autorange='reversed')  # rank 1 at top
fig.show()

# headline robustness statement
robust5  = mc[mc['pct_top5']  >= 80]['iso_code'].tolist()
robust10 = mc[mc['pct_top10'] >= 90]['iso_code'].tolist()
print(f"\nIn the top-5 under >=80% of ALL random weightings: {robust5}")
print(f"In the top-10 under >=90% of ALL random weightings: {robust10}")

Monte Carlo sensitivity - 2000 random weightings (2000 base, clean):
iso_code  median_rank  best  worst  pct_top5  pct_top10
     SWE            2     1      6      99.9      100.0
     FIN            3     1     21      87.3       98.4
     NOR            3     1     20      74.3       92.3
     PRT            4     1     16      76.8       94.0
     GBR            5     1     15      65.3       97.6
     DNK            6     1     23      39.1       88.8
     FRA            7     3     14      23.4       82.7
     ESP            8     3     13      10.0       89.1
     EST           10     3     28       6.2       59.6
     DEU           11     4     27       2.1       41.6
     AUT           11     5     25       0.5       40.2
     GRC           11     4     23       1.5       49.6



In the top-5 under >=80% of ALL random weightings: ['SWE', 'FIN']
In the top-10 under >=90% of ALL random weightings: ['SWE', 'FIN', 'NOR', 'PRT', 'GBR']


In [13]:
# FAIRNESS REFEREE 3: rank-correlation + leave-one-component-out
from scipy.stats import spearmanr

# (a) How similar are the rankings across the named scenarios? (1.0 = identical order)
rank_df = pd.DataFrame(rank_table)
corr = rank_df.corr(method='spearman')
off_diag = corr.values[np.triu_indices_from(corr.values, k=1)]
print("Spearman rank-correlation between weighting scenarios:")
print(corr.round(2).to_string())
print(f"\nMean off-diagonal correlation: {off_diag.mean():.3f}  "
      f"(min {off_diag.min():.3f}) - close to 1.0 means the order barely depends on weights")

# (b) Leave-one-component-out: drop each component entirely, renormalize, compare to normative.
#     Low Spearman = that component is 'load-bearing' (the ranking leans on it).
normative_rank = rank_with_weights(base_clean, WEIGHTS)
isos = base_clean['iso_code'].tolist()
norm_top5 = set(sorted(normative_rank, key=normative_rank.get)[:5])

print("\nLeave-one-component-out (drop a component, renormalize the other four):")
print(f"{'dropped':<16}{'spearman vs normative':>22}{'  top-5 change'}")
for drop in WEIGHTS:
    w = {k: v for k, v in WEIGHTS.items() if k != drop}
    print(w)
    print(base_clean)
    s = sum(w.values()) 
    w = {k: v / s for k, v in w.items()}
    r = rank_with_weights(base_clean, w)
    rho = spearmanr([normative_rank[i] for i in isos], [r[i] for i in isos]).correlation
    new_top5 = set(sorted(r, key=r.get)[:5])
    changed = norm_top5.symmetric_difference(new_top5)
    print(f"{drop:<16}{rho:>22.3f}   {sorted(changed) if changed else 'none'}")

Spearman rank-correlation between weighting scenarios:
                   Normative  Equal  Honesty-heavy  Destination-heavy  Prosperity-heavy  Reduction-heavy
Normative               1.00   0.96           0.99               0.97              0.92             0.91
Equal                   0.96   1.00           0.93               0.96              0.96             0.96
Honesty-heavy           0.99   0.93           1.00               0.95              0.88             0.88
Destination-heavy       0.97   0.96           0.95               1.00              0.90             0.89
Prosperity-heavy        0.92   0.96           0.88               0.90              1.00             0.96
Reduction-heavy         0.91   0.96           0.88               0.89              0.96             1.00

Mean off-diagonal correlation: 0.934  (min 0.880) - close to 1.0 means the order barely depends on weights

Leave-one-component-out (drop a component, renormalize the other four):
dropped          spearman vs 

KeyError: 'honesty'

In [ ]:
# FAIRNESS REFEREE 4: threshold robustness
# Weights are not the only subjective choice - the SCORING anchors are too.
# Jitter every threshold by +-15% many times and check the ranking still holds.
import copy

rng_t = np.random.default_rng(7)
TIMES = 200
FRACTION = 0.15

def perturb_scoring(base, frac, rng):
    p = copy.deepcopy(base)
    for k, d in p.items():
        for nail in ('zero', 'full'):
            p[k][nail] = d[nail] * (1 + rng.uniform(-frac, frac))
    return p

orig_SC = SC                     # SC is the global SCORING dict that build_scores reads
thr_hist = {iso: [] for iso in base_clean['iso_code']}

for _ in range(TIMES):
    SC = perturb_scoring(orig_SC, FRACTION, rng_t)   # reassign global -> build_scores picks it up
    s = build_scores(df, 2000)
    s = s[s['flag'] == ''].reset_index(drop=True)
    for i, row in s.iterrows():
        thr_hist[row['iso_code']].append(i + 1)
SC = orig_SC 

thr_rows = []
for iso, ranks in thr_hist.items():
    ranks = np.array(ranks)
    thr_rows.append({'iso_code': iso, 'median_rank': int(np.median(ranks)),
                     'best': int(ranks.min()), 'worst': int(ranks.max()),
                     'pct_top5': round((ranks <= 5).mean() * 100, 1)})
thr = pd.DataFrame(thr_rows).sort_values('median_rank').reset_index(drop=True)

print(f"Threshold robustness - {TIMES} runs, every anchor jittered +-{int(FRACTION*100)}%:")
print(thr.head(12).to_string(index=False))
print("\n(If the top names here match the weight-based Monte Carlo, the ranking is robust")
print(" to BOTH subjective choices - weights AND thresholds.)")

Threshold robustness - 200 = 200 runs, every anchor jittered +-15%:
iso_code  median_rank  best  worst  pct_top5
     SWE            1     1      3     100.0
     PRT            2     1      4     100.0
     FIN            3     2      4     100.0
     NOR            4     2      4     100.0
     GBR            5     5      5     100.0
     EST            6     6      6       0.0
     DNK            7     7     10       0.0
     DEU            8     7      9       0.0
     ESP            9     7     11       0.0
     AUT           10     7     11       0.0
     GRC           11     9     12       0.0
     FRA           12    11     12       0.0

(If the top names here match the weight-based Monte Carlo, the ranking is robust
 to BOTH subjective choices - weights AND thresholds.)


In [ ]:
# ---------- FAIRNESS REFEREE 5: component redundancy (construct validity) ----------
# If two components correlate strongly, they double-count - one is partly redundant.
comp_score_cols = ['s_honesty', 's_energy', 's_abs', 's_prosperity', 's_co2']
nice = {'s_honesty': 'honesty', 's_energy': 'energy', 's_abs': 'abs_cons',
        's_prosperity': 'prosperity', 's_co2': 'co2_reduc'}

cc = scores_2000[comp_score_cols].corr().rename(index=nice, columns=nice)
print("Component score correlation (|r|>0.6 = possible double-counting):")
print(cc.round(2).to_string())

fig = px.imshow(cc, text_auto='.2f', color_continuous_scale='RdBu_r', zmin=-1, zmax=1,
                title='Component correlation - are the 5 components measuring distinct things?',
                height=500)
fig.show()

pairs = [(nice[a], nice[b], cc.iloc[i, j])
         for i, a in enumerate(comp_score_cols)
         for j, b in enumerate(comp_score_cols) if j > i]
redundant = [(a, b, round(v, 2)) for a, b, v in pairs if abs(v) > 0.6]
print("\nStrongly-correlated pairs (|r|>0.6):", redundant if redundant else "none - components are distinct")

Component score correlation (|r|>0.6 = possible double-counting):
            honesty  energy  abs_cons  prosperity  co2_reduc
honesty        1.00    0.01      0.25        0.33       0.43
energy         0.01    1.00     -0.42        0.51       0.40
abs_cons       0.25   -0.42      1.00       -0.52      -0.44
prosperity     0.33    0.51     -0.52        1.00       0.56
co2_reduc      0.43    0.40     -0.44        0.56       1.00



Strongly-correlated pairs (|r|>0.6): none - components are distinct


In [ ]:
print("=" * 30)
print("NOTEBOOK 08 - KEY FINDINGS")
print("=" * 30)

top10 = scores_2000.head(10)
print("\nTop 10 decouplers in Europe (2000 base, normative weights):")
for i, (_, r) in enumerate(top10.iterrows(), 1):
    print(f"  {i:>2}. {r['country']:<16} {r['final']:6.1f}")

w = scores_2000.iloc[0]
print(f"\nLeader: {w['country']} ({w['final']:.1f})")
print(f"   territorial -{w['co2_red_pct']:.0f}%  |  consumption -{w['cons_red_pct']:.0f}%  |  "
      f"GDP/cap +{w['gdp_grw_pct']:.0f}% (${w['gdp_latest']:,.0f})  |  consumption {w['cons_latest']:.1f} t/cap")

# face-validity: Switzerland should sink on honesty (can go negative)
for iso in ['CHE', 'POL', 'SWE', 'DEU']:
    if (scores_2000['iso_code'] == iso).any():
        r = scores_2000[scores_2000['iso_code'] == iso].iloc[0]
        rank = scores_2000.index[scores_2000['iso_code'] == iso][0] + 1
        print(f"   check {iso}: rank {rank:>2}, honesty {r['s_honesty']:6.1f}, "
              f"cons.reduction {r['cons_red_pct']:5.0f}%, divergence {r['divergence']:5.0f}")

print("""
Narrative to fill after review:
- Did Sweden / Germany / Finland climb back up vs the first version?
- Did flat-emissions Poland drop?
- Does Switzerland now show a NEGATIVE honesty score?
- Journey vs destination: any high-score country still sitting at high consumption?
- Did the Top-10 survive the sensitivity analysis (rank_range)?
""")

NOTEBOOK 08 - KEY FINDINGS

Top 10 decouplers in Europe (2000 base, normative weights):
   1. Sweden             82.6
   2. Portugal           82.2
   3. Finland            81.9
   4. Norway             80.5
   5. Ireland            79.3
   6. Luxembourg         77.7
   7. United Kingdom     76.5
   8. Estonia            72.6
   9. Denmark            68.1
  10. Germany            67.4

Leader: Sweden (82.6)
   territorial -44%  |  consumption -38%  |  GDP/cap +38% ($47,125)  |  consumption 5.8 t/cap
   check CHE: rank 30, honesty  -66.5, cons.reduction    -6%, divergence    46
   check POL: rank 20, honesty   28.9, cons.reduction     8%, divergence     4
   check SWE: rank  1, honesty   82.4, cons.reduction    38%, divergence     5
   check DEU: rank 10, honesty   75.2, cons.reduction    32%, divergence     4

Narrative to fill after review:
- Did Sweden / Germany / Finland climb back up vs the first version?
- Did flat-emissions Poland drop?
- Does Switzerland now show a NEGATIVE hone

In [ ]:
# ---------- TRANSPARENCY: per-country scorecard ----------
# Trace exactly how any country's score is built: raw input -> points -> weight -> contribution.

def explain(iso, scores=None, base_year=2000):
    scores = scores_2000 if scores is None else scores
    sub = scores[scores['iso_code'] == iso]
    if sub.empty:
        print(f"{iso} not in ranking"); return
    r = sub.iloc[0]
    rank = scores.index[scores['iso_code'] == iso][0] + 1

    print(f"=== {r['country']} ({iso}) - rank {rank}, FINAL {r['final']:.1f}  (base {base_year}) ===")
    if r['flag']:
        print(f"  ! data-quality flag: {DATA_CAVEAT.get(iso, '')}")
    print(f"{'component':<16}{'raw input':>36}{'pts':>7}{'weight':>8}{'contrib':>8}")
    print('-' * 75)

    rows = [
        ('honesty',         f"cons -{r['cons_red_pct']:.0f}% | recent -{r['cons_red_recent_pct']:.0f}% | div {r['divergence']:+.0f}", r['s_honesty']),
        ('co2_reduction',   f"territorial -{r['co2_red_pct']:.0f}%",                  r['s_co2']),
        ('abs_consumption', f"{r['cons_latest']:.1f} t/cap now",                      r['s_abs']),
        ('energy_clean',    f"{r['en_latest']:.3f} kgCO2/kWh now",                    r['s_energy']),
        ('prosperity',      f"${r['gdp_latest']:,.0f}/cap | growth +{r['gdp_grw_pct']:.0f}%", r['s_prosperity']),
    ]
    avail_w = sum(WEIGHTS[k] for k, _, pts in rows if pd.notna(pts))
    for name, raw, pts in rows:
        if pd.isna(pts):
            print(f"{name:<16}{raw:>36}{'n/a':>7}{'-':>8}{'-':>8}")
            continue
        eff_w = WEIGHTS[name] / avail_w  # weight actually used (renormalized if a component is n/a)
        print(f"{name:<16}{raw:>36}{pts:>7.1f}{eff_w:>8.2f}{pts*eff_w:>8.1f}")
    print('-' * 75)
    print(f"{'composite':<16}{'(weighted sum above)':>36}{'':>7}{'':>8}{r['composite']:>8.1f}")
    print(f"{'momentum':<16}{'(consumption slope 2010-latest)':>36}{'':>7}{'':>8}{r['momentum']:>+8.1f}")
    print(f"{'FINAL':<16}{'':>36}{'':>7}{'':>8}{r['final']:>8.1f}")

# A star, a fake decoupler, and the standout-that-feels-underrated
for code in ['SWE', 'CHE', 'ROU']:
    explain(code)
    print()

=== Sweden (SWE) - rank 1, FINAL 85.5  (base 2000) ===
component                                  raw input    pts  weight contrib
---------------------------------------------------------------------------
honesty             cons -38% | recent -34% | div +5   87.1    0.40    34.8
co2_reduction                       territorial -44%   72.9    0.15    10.9
abs_consumption                        5.8 t/cap now   72.5    0.15    10.9
energy_clean                     0.061 kgCO2/kWh now   95.6    0.15    14.3
prosperity                 $47,125/cap | growth +38%   82.8    0.15    12.4
---------------------------------------------------------------------------
composite                       (weighted sum above)                   83.4
momentum             (consumption slope 2010-latest)                   +2.1
FINAL                                                                  85.5

=== Switzerland (CHE) - rank 30, FINAL 10.2  (base 2000) ===
component                                  raw 